In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Oprócz [wizualizacji instrukcji w obwodzie](/guides/visualize-circuits) możesz chcieć zwizualizować harmonogram obwodu za pomocą metody Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer). Ta wizualizacja może pomóc na przykład szybko wykryć czas bezczynności na qubitach. Jednak ta metoda nie zwraca dokładnych wyników dla obwodów dynamicznych. Aby zwizualizować harmonogram obwodów dynamicznych, użyj metody `draw_circuit_schedule_timing`, jak opisano w sekcji [Wsparcie Qiskit Runtime](#qr-support).

## Przykłady

Aby zwizualizować zaplanowany program obwodu, możesz wywołać tę funkcję z zestawem argumentów sterujących. Większość wyglądu obrazu wyjściowego można modyfikować za pomocą arkusza stylów, ale nie jest to wymagane.

### Rysowanie z domyślnym arkuszem stylów

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Rysowanie z arkuszem stylów przystosowanym do debugowania programu

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Możesz tworzyć niestandardowe funkcje generatora lub układu i aktualizować istniejący arkusz stylów za pomocą tych niestandardowych funkcji. W ten sposób możesz kontrolować większość wyglądu obrazu wyjściowego bez modyfikowania bazy kodu rysownika zaplanowanych obwodów. Zobacz dokumentację API [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer), aby uzyskać więcej przykładów.
<span id="qr-support"></span>
## Wsparcie Qiskit Runtime
Chociaż rysownik osi czasu wbudowany w Qiskit jest przydatny dla obwodów statycznych, może nie odzwierciedlać dokładnie taktowania [obwodów dynamicznych](/guides/classical-feedforward-and-control-flow) z powodu operacji niejawnych, takich jak rozgłaszanie i wyznaczanie gałęzi. W ramach wsparcia obwodów dynamicznych Qiskit Runtime zwraca dokładne informacje o taktowaniu obwodu w wynikach zadania, gdy są one żądane.

> **Note:** - Jest to funkcja eksperymentalna. Ma status wersji zapoznawczej i może ulec zmianie.
> - Ta funkcja dotyczy tylko zadań Sampler w Qiskit Runtime.
> - Chociaż całkowity czas obwodu jest zwracany w metadanych "compilation", NIE jest to czas używany do rozliczeń (czas kwantowy).
### Włączanie pobierania danych taktowania
Aby włączyć pobieranie danych taktowania, ustaw eksperymentalną flagę `scheduler_timing` na `True` podczas uruchamiania zadania prymitywu.